In [1]:
import json
import os
import sys
from pathlib import Path

os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3"

import torch

from satchange import BCEDiceLoss
from satchange.data import (
    RandomFlipRotate,
    create_dataset_dataloader,
    prepare_levir_cd,
)
from satchange.experiments import load_fc_siam_diff_experiment
from satchange.models import FCSiamDiff
from satchange.optimization import run_parameter_search

In [2]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

Device: cuda


In [3]:
print("Python executable:", sys.executable)
print("Python version:", sys.version)

print("PyTorch:", torch.__version__)
print("PyTorch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("Visible GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("Selected GPU:", torch.cuda.get_device_name(0))

Python executable: /home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/.venv/bin/python
Python version: 3.11.6 | packaged by conda-forge | (main, Oct  3 2023, 10:40:35) [GCC 12.3.0]
PyTorch: 2.12.1+cu130
PyTorch CUDA: 13.0
CUDA available: True
Visible GPU count: 4
Selected GPU: NVIDIA A100-SXM4-80GB


In [4]:
current_directory = Path.cwd()

if current_directory.name == "notebooks":
    project_root = current_directory.parent
else:
    project_root = current_directory

full_config_path = (
    project_root
    / "configs"
    / "fc_siam_diff_full.yaml"
)

resized_config_path = (
    project_root
    / "configs"
    / "fc_siam_diff_resized.yaml"
)


print("The project root: ", project_root)

print("Full config:", full_config_path)
print("Full config exists:", full_config_path.exists())

print("Resized config:", resized_config_path)
print("Resized config exists:", resized_config_path.exists())

The project root:  /home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence
Full config: /home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/configs/fc_siam_diff_full.yaml
Full config exists: True
Resized config: /home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/configs/fc_siam_diff_resized.yaml
Resized config exists: True


### Resized Dataset

In [5]:
EXPERIMENT = "resized"
# Available "full" and "resized"


config, config_path = load_fc_siam_diff_experiment(
    experiment=EXPERIMENT,
    project_root=project_root,
)

print("Experiment:", EXPERIMENT)
print("Configuration:", config_path)
print("Image size:", config["data"]["image_size"])
print("Batch size:", config["data"]["batch_size"])
print("Output directory:", config["output"]["directory"])

Experiment: resized
Configuration: /home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/configs/fc_siam_diff_resized.yaml
Image size: 256
Batch size: 16
Output directory: artifacts/fc_siam_diff_resized


In [6]:
RUN_DATA_SETUP = False
dataset_root = project_root / config["dataset"]["prepared_directory"]

if RUN_DATA_SETUP:
    dataset_root = prepare_levir_cd(
        config=config,
        project_root=project_root,
    )
    print("Ready dataset root:", dataset_root)
else:
    print("Dataset preparation skipped; set RUN_DATA_SETUP=True to enable it.")

Download directory is not empty: /home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/data/downloads/levir_cd
Skipping download.
Split is already extracted: /home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/data/LEVIR-CD/train
Skipping extraction.
Split is already extracted: /home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/data/LEVIR-CD/val
Skipping extraction.
Split is already extracted: /home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/data/LEVIR-CD/test
Skipping extraction.

Split:         train
Before images: 445
After images:  445
Labels:        445

Split:         val
Before images: 64
After images:  64
Labels:        64

Split:         test
Before images: 128
After images:  128
Labels:        128

Total samples: 637
LEVIR-CD structure is valid.
Ready dataset root: /home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/data/LEVIR-CD


In [7]:
if RUN_DATA_SETUP:
    train_transform = RandomFlipRotate(
        horizontal_flip_probability=0.5,
        vertical_flip_probability=0.5,
    )
    train_dataset, train_loader = create_dataset_dataloader(
        dataset_root=dataset_root,
        split="train",
        image_size=config["data"]["image_size"],
        batch_size=config["data"]["batch_size"],
        num_workers=config["data"]["num_workers"],
        transform=train_transform,
        shuffle=True,
        pin_memory=torch.cuda.is_available(),
        seed=config["project"]["seed"],
    )
    validation_dataset, validation_loader = create_dataset_dataloader(
        dataset_root=dataset_root,
        split="val",
        image_size=config["data"]["image_size"],
        batch_size=config["data"]["batch_size"],
        num_workers=config["data"]["num_workers"],
        pin_memory=torch.cuda.is_available(),
    )
    test_dataset, test_loader = create_dataset_dataloader(
        dataset_root=dataset_root,
        split="test",
        image_size=config["data"]["image_size"],
        batch_size=config["data"]["batch_size"],
        num_workers=config["data"]["num_workers"],
        pin_memory=torch.cuda.is_available(),
    )
else:
    train_dataset = train_loader = None
    validation_dataset = validation_loader = None
    test_dataset = test_loader = None
    print("Loader construction skipped with dataset setup.")

In [8]:
RANDOM_SEED = config["project"]["seed"]

torch.manual_seed(RANDOM_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(
        RANDOM_SEED
    )


model = FCSiamDiff(
    in_channels=(
        config["model"]["parameters"]["in_channels"]
    ),
    out_channels=(
        config["model"]["parameters"]["out_channels"]
    ),
    base_channels=(
        config["model"]["parameters"]["base_channels"]
    ),
).to(device)


loss_function = BCEDiceLoss(
    bce_weight=(
        config["training"]["loss"]["bce_weight"]
    ),
    dice_weight=(
        config["training"]["loss"]["dice_weight"]
    ),
    pos_weight=5,
).to(device)


optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config["training"]["learning_rate"],
    weight_decay=(
        config["training"]["weight_decay"]
    ),
)

In [9]:
THRESHOLDS = [
    round(value / 100, 2)
    for value in range(5, 96, 5)
]

INSPECT_RESUME_CHECKPOINT = False
batch_size = config["data"]["batch_size"]
learning_rate = config["training"]["learning_rate"]
seed = RANDOM_SEED
positive_weight = 10

MODEL_NAME = "fc_siam_diff"

RUN_NAME = (
    f"levir_{EXPERIMENT}_"
    f"pw{positive_weight:g}_"
    f"lr{learning_rate:g}_"
    f"bs{batch_size}_"
    f"seed{seed}"
)

OUTPUT_ROOT = (
    project_root
    / "artifacts"
    / "training_runs"
)

RUN_DIRECTORY = (
    OUTPUT_ROOT
    / MODEL_NAME
    / RUN_NAME
)

RESUME_CHECKPOINT = (
    RUN_DIRECTORY
    / "last.pt"
)

print("Run name:", RUN_NAME)
print("Resume checkpoint:", RESUME_CHECKPOINT)

if INSPECT_RESUME_CHECKPOINT:
    if not RESUME_CHECKPOINT.is_file():
        raise FileNotFoundError(
            f"Resume checkpoint was not found: {RESUME_CHECKPOINT}"
        )
    checkpoint_information = torch.load(
        RESUME_CHECKPOINT,
        map_location="cpu",
        weights_only=False,
    )
    print("Saved epoch:", checkpoint_information["epoch"])
    print("Saved model:", checkpoint_information.get("model_name"))
    print("Saved run:", checkpoint_information.get("run_name"))
    print("Best metrics:", checkpoint_information.get("best_metrics"))
    print(
        "Contains optimizer state:",
        "optimizer_state_dict" in checkpoint_information,
    )
else:
    print("Checkpoint inspection skipped.")

Run name: levir_resized_pw10_lr0.001_bs16_seed42
Resume checkpoint: /home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/artifacts/training_runs/fc_siam_diff/levir_resized_pw10_lr0.001_bs16_seed42/last.pt
Saved epoch: 110
Saved model: fc_siam_diff
Saved run: levir_resized_pw10_lr0.001_bs16_seed42
Best metrics: {'epoch': 103, 'train_loss': 0.36003440692183675, 'validation_loss': 0.36435065418481827, 'best_threshold': 0.85, 'validation_precision': 0.7088226011818607, 'validation_recall': 0.74000056806885, 'validation_f1': 0.7240761173164949, 'validation_iou': 0.5674916248088627, 'validation_accuracy': 0.9763295650482178, 'true_positive': 130266, 'false_positive': 53512, 'false_negative': 45769, 'true_negative': 3964757, 'f1_at_05': 0.6830171111425533, 'mean_changed_probability': 0.8214275348297214, 'mean_unchanged_probability': 0.03312555253502814, 'probability_gap': 0.7883019822946933}
Contains optimizer state: True


In [13]:
SEARCH_SPACE = {
    "weight_decay": [
        1e-5,
        1e-4,
        1e-3,
    ],
}

FIXED_PARAMETERS = {
    "learning_rate": 1e-3,
    "scheduler": "none",
    "positive_weight": 10,
    "bce_weight": 0.3,
    "dice_weight": 0.7,

    "batch_size": config["data"]["batch_size"],
    "image_size": config["data"]["image_size"],
    "num_workers": config["data"]["num_workers"],
    "seed": config["project"]["seed"],
    "model_parameters": dict(
        config["model"]["parameters"]
    ),
}


RUN_PARAMETER_SEARCH = False
results_path = (
    project_root
    / "artifacts"
    / "hyperparameter_search"
    / "stage3_weight_decay"
    / "results.json"
)

if RUN_PARAMETER_SEARCH:
    results = run_parameter_search(
        search_space=SEARCH_SPACE,
        fixed_parameters=FIXED_PARAMETERS,
        model_name="fc_siam_diff",
        dataset_root=dataset_root,
        thresholds=THRESHOLDS,
        number_of_epochs=60,
        output_root=project_root / "artifacts" / "hyperparameter_search",
        search_name="stage3_weight_decay",
        gpu_ids=[0, 1, 2, 3],
        checkpoint_interval=5,
    )
else:
    with results_path.open(encoding="utf-8") as results_file:
        results = json.load(results_file)["results"]
    print("Parameter search skipped; loaded:", results_path)

Trials: 3
GPUs: [0, 1, 2, 3]



[Trial 001 | GPU 1] Epoch 1/60
Train loss:       0.920112
Validation loss:  0.904386
Best threshold:   0.35
Precision:        0.069238
Recall:           0.515011
F1:               0.122066
IoU:              0.065000

[Trial 002 | GPU 2] Epoch 1/60
Train loss:       0.920124
Validation loss:  0.904512
Best threshold:   0.35
Precision:        0.067941
Recall:           0.519766
F1:               0.120174
IoU:              0.063928

[Trial 000 | GPU 0] Epoch 1/60
Train loss:       0.920145
Validation loss:  0.904670
Best threshold:   0.35
Precision:        0.064835
Recall:           0.552799
F1:               0.116058
IoU:              0.061604

[Trial 002 | GPU 2] Epoch 2/60
Train loss:       0.902461
Validation loss:  0.878479
Best threshold:   0.50
Precision:        0.153254
Recall:           0.249638
F1:               0.189917
IoU:              0.104921

[Trial 001 | GPU 1] Epoch 2/60
Train loss:       0.901925
Validation loss:  0.865790
Best threshold:   0.45
Precision:        0.137

In [14]:
# Sort all trials from best to worst by validation F1.
sorted_results = sorted(
    results,
    key=lambda result: result["best_f1"],
    reverse=True,
)

best_result = sorted_results[0]
best_parameters = best_result["parameters"]


# ---------------------------------------------------------------------
# Full Stage 2 ranking
# ---------------------------------------------------------------------

print()
print("=" * 115)
print("STAGE 2: POSITIVE WEIGHT × BCE/DICE LOSS WEIGHTS")
print("=" * 115)

print(
    f"{'Trial':>5} | "
    f"{'Pos W':>6} | "
    f"{'BCE':>5} | "
    f"{'Dice':>5} | "
    f"{'Epoch':>5} | "
    f"{'Thresh':>6} | "
    f"{'Precision':>9} | "
    f"{'Recall':>9} | "
    f"{'F1':>9} | "
    f"{'IoU':>9} | "
    f"{'GPU h':>7}"
)

print("-" * 115)

for result in sorted_results:
    parameters = result["parameters"]

    print(
        f"{result['trial']:5d} | "
        f"{parameters['positive_weight']:6g} | "
        f"{parameters['bce_weight']:5.2f} | "
        f"{parameters['dice_weight']:5.2f} | "
        f"{result['best_epoch']:5d} | "
        f"{result['best_threshold']:6.2f} | "
        f"{result['best_precision']:9.4f} | "
        f"{result['best_recall']:9.4f} | "
        f"{result['best_f1']:9.4f} | "
        f"{result['best_iou']:9.4f} | "
        f"{result['gpu_hours']:7.3f}"
    )


# ---------------------------------------------------------------------
# Best overall trial
# ---------------------------------------------------------------------

print()
print("=" * 80)
print("BEST STAGE 3 TRIAL")
print("=" * 80)

print(
    f"Trial:             {best_result['trial']}\n"
    f"Positive weight:   {best_parameters['positive_weight']}\n"
    f"BCE weight:        {best_parameters['bce_weight']:.2f}\n"
    f"Dice weight:       {best_parameters['dice_weight']:.2f}\n"
    f"Best epoch:        {best_result['best_epoch']}\n"
    f"Best threshold:    {best_result['best_threshold']:.2f}\n"
    f"Precision:         {best_result['best_precision']:.6f}\n"
    f"Recall:            {best_result['best_recall']:.6f}\n"
    f"F1:                {best_result['best_f1']:.6f}\n"
    f"IoU:               {best_result['best_iou']:.6f}\n"
    f"Validation loss:   {best_result['best_validation_loss']:.6f}\n"
    f"Training time:     {best_result['training_seconds'] / 60:.2f} min\n"
    f"GPU hours:         {best_result['gpu_hours']:.3f}\n"
    f"Peak GPU memory:   {best_result['peak_gpu_memory_mb']:.1f} MB"
)


# ---------------------------------------------------------------------
# Best result for each positive weight
# ---------------------------------------------------------------------

print()
print("=" * 80)
print("BEST RESULT FOR EACH POSITIVE WEIGHT")
print("=" * 80)

for positive_weight in sorted(
    {
        result["parameters"]["positive_weight"]
        for result in results
    }
):
    candidates = [
        result
        for result in results
        if result["parameters"]["positive_weight"]
        == positive_weight
    ]

    best = max(
        candidates,
        key=lambda result: result["best_f1"],
    )

    parameters = best["parameters"]

    print(
        f"Pos weight {positive_weight:g} | "
        f"BCE/Dice {parameters['bce_weight']:.1f}/"
        f"{parameters['dice_weight']:.1f} | "
        f"F1 {best['best_f1']:.6f} | "
        f"IoU {best['best_iou']:.6f} | "
        f"Epoch {best['best_epoch']}"
    )


# ---------------------------------------------------------------------
# Best result for each BCE/Dice combination
# ---------------------------------------------------------------------

print()
print("=" * 80)
print("BEST RESULT FOR EACH BCE/DICE COMBINATION")
print("=" * 80)

loss_combinations = sorted(
    {
        (
            result["parameters"]["bce_weight"],
            result["parameters"]["dice_weight"],
        )
        for result in results
    }
)

for bce_weight, dice_weight in loss_combinations:
    candidates = [
        result
        for result in results
        if (
            result["parameters"]["bce_weight"]
            == bce_weight
            and result["parameters"]["dice_weight"]
            == dice_weight
        )
    ]

    best = max(
        candidates,
        key=lambda result: result["best_f1"],
    )

    print(
        f"BCE/Dice {bce_weight:.1f}/{dice_weight:.1f} | "
        f"Pos weight "
        f"{best['parameters']['positive_weight']:g} | "
        f"F1 {best['best_f1']:.6f} | "
        f"IoU {best['best_iou']:.6f} | "
        f"Epoch {best['best_epoch']}"
    )


# ---------------------------------------------------------------------
# Compute cost
# ---------------------------------------------------------------------

total_gpu_hours = sum(
    result["gpu_hours"]
    for result in results
)

total_training_seconds = sum(
    result["training_seconds"]
    for result in results
)

average_seconds_per_epoch = sum(
    result["seconds_per_epoch"]
    for result in results
) / len(results)

maximum_peak_gpu_memory_mb = max(
    result["peak_gpu_memory_mb"]
    for result in results
)

print()
print("=" * 80)
print("COMPUTE COST")
print("=" * 80)

print(
    f"Trials:                  {len(results)}\n"
    f"Total GPU hours:         {total_gpu_hours:.3f}\n"
    f"Sum of training time:    {total_training_seconds / 3600:.3f} h\n"
    f"Average sec/epoch:       {average_seconds_per_epoch:.2f}\n"
    f"Maximum peak GPU memory: {maximum_peak_gpu_memory_mb:.1f} MB"
)


# ---------------------------------------------------------------------
# Parameters selected for Stage 3
# ---------------------------------------------------------------------

STAGE_2_BEST = {
    "learning_rate": best_parameters["learning_rate"],
    "scheduler": best_parameters["scheduler"],
    "positive_weight": best_parameters["positive_weight"],
    "bce_weight": best_parameters["bce_weight"],
    "dice_weight": best_parameters["dice_weight"],
}

print()
print("=" * 80)
print("PARAMETERS SELECTED FOR STAGE 3")
print("=" * 80)

for name, value in STAGE_2_BEST.items():
    print(f"{name}: {value}")


STAGE 2: POSITIVE WEIGHT × BCE/DICE LOSS WEIGHTS
Trial |  Pos W |   BCE |  Dice | Epoch | Thresh | Precision |    Recall |        F1 |       IoU |   GPU h
-------------------------------------------------------------------------------------------------------------------
    1 |     10 |  0.30 |  0.70 |    59 |   0.85 |    0.6783 |    0.7484 |    0.7116 |    0.5523 |   1.805
    2 |     10 |  0.30 |  0.70 |    60 |   0.90 |    0.6443 |    0.7635 |    0.6989 |    0.5371 |   1.803
    0 |     10 |  0.30 |  0.70 |    57 |   0.90 |    0.6645 |    0.6996 |    0.6816 |    0.5170 |   1.808

BEST STAGE 2 TRIAL
Trial:             1
Positive weight:   10
BCE weight:        0.30
Dice weight:       0.70
Best epoch:        59
Best threshold:    0.85
Precision:         0.678306
Recall:            0.748385
F1:                0.711625
IoU:               0.552343
Validation loss:   0.398570
Training time:     108.29 min
GPU hours:         1.805
Peak GPU memory:   1595.7 MB

BEST RESULT FOR EACH POSITIV